# 02. 학습 + 평가 (All-in-One)
Flow Matching 학습 → 샘플링 → 돌연변이 효과 분석 → 응집 상관관계

In [ ]:
# 환경 설정
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm tensorboard scipy -q
sys.path.insert(0, '/content/brain_idp_flow/src')

import torch
import yaml
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# 데이터 준비 (임베딩 + 데이터셋)
os.makedirs('data', exist_ok=True)

from brain_idp_flow.targets import load_targets
from brain_idp_flow.features.seq_embed import ESM2Embedder

targets = load_targets('configs/targets.yaml')
embedder = ESM2Embedder(device=device)

seq_embeddings = {}
sid = 0
for tid, target in targets.items():
    emb = embedder.embed_single(target.sequence)
    seq_embeddings[sid] = emb.cpu()
    print(f'{tid} WT: seq_id={sid}, emb={emb.shape}')
    sid += 1
    for mut in target.mutations:
        emb = embedder.embed_single(target.mutant_sequence(mut))
        seq_embeddings[sid] = emb.cpu()
        print(f'  {mut.id}: seq_id={sid}')
        sid += 1

torch.save(seq_embeddings, 'data/seq_embeddings.pt')
print(f'\n{len(seq_embeddings)} embeddings saved')

if not os.path.exists('data/train.npz'):
    !python scripts/build_dataset.py --out data/train.npz --max-len 160
print('Data ready')

In [ ]:
# Flow Matching 학습
from brain_idp_flow.train import train

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

seq_embeddings = torch.load('data/seq_embeddings.pt', weights_only=True)
best_ckpt = train(config, seq_embeddings, device)
print(f'\nBest checkpoint: {best_ckpt}')

In [ ]:
# Google Drive 백업
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
backup_dir = '/content/drive/MyDrive/brain_idp_flow_ckpts'
os.makedirs(backup_dir, exist_ok=True)
for f in ['ckpt_best.pt', 'ckpt_last.pt']:
    src = str(best_ckpt).replace('ckpt_best.pt', f)
    if os.path.exists(src):
        shutil.copy2(src, backup_dir)
        print(f'Backed up {src}')

---
# 평가 시작

In [ ]:
# 모델 로드 (방금 학습한 체크포인트 사용)
from brain_idp_flow.sample import load_model, sample_ensemble
from brain_idp_flow.data.dataset import ProteinEnsembleDataset

model = load_model(config, best_ckpt, device)
print(f'Model loaded from {best_ckpt}')

In [ ]:
# WT + 돌연변이 앙상블 생성
os.makedirs('samples/flow', exist_ok=True)
os.makedirs('samples/eval', exist_ok=True)

for tid, target in targets.items():
    print(f'\n=== {target.name} ===')
    
    seq_emb = embedder.embed_single(target.sequence)
    wt = sample_ensemble(model, seq_emb, n_samples=200, device=device)
    np.save(f'samples/flow/{tid}_WT.npy', wt)
    print(f'  WT: {wt.shape}')
    
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        mut_emb = embedder.embed_single(mut_seq)
        aa_idx = ProteinEnsembleDataset.AA_TO_IDX.get(mut.mt, 0)
        
        mut_ens = sample_ensemble(
            model, mut_emb, mut_pos=mut.pos, mut_aa=aa_idx,
            n_samples=200, device=device
        )
        np.save(f'samples/flow/{tid}_{mut.id}.npy', mut_ens)
        print(f'  {mut.id}: {mut_ens.shape}')

In [ ]:
# PED 대비 정량 평가
from brain_idp_flow.data.ped_loader import load_ped_or_fallback
from brain_idp_flow.eval.compare import compare_ensembles, compare_mutation_effect

print('=== WT vs PED Reference ===')
for tid, target in targets.items():
    wt = np.load(f'samples/flow/{tid}_WT.npy')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    metrics = compare_ensembles(wt, ref)
    print(f'{tid}: {metrics.summary()}')

In [ ]:
# 돌연변이 효과 분석
print('\n=== Mutation Effects ===')
mutation_results = []

for tid, target in targets.items():
    wt = np.load(f'samples/flow/{tid}_WT.npy')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    
    for mut in target.mutations:
        mut_path = f'samples/flow/{tid}_{mut.id}.npy'
        if not os.path.exists(mut_path):
            continue
        mut_ens = np.load(mut_path)
        effect = compare_mutation_effect(wt, mut_ens, ref)
        effect['target'] = tid
        effect['mutation'] = mut.id
        effect['agg_rate'] = mut.agg_rate_relative
        mutation_results.append(effect)
        
        print(f'{tid} {mut.id}: \u0394Rg={effect["delta_rg_mean"]:+.2f}  '
              f'JS={effect["rg_js_wt_vs_mut"]:.4f}  '
              f'agg_rate={mut.agg_rate_relative}x')

In [ ]:
# 핵심 플롯 1: Rg 분포 (WT vs 돌연변이 vs PED)
from brain_idp_flow.eval.plots import plot_mutation_comparison

for tid, target in targets.items():
    wt = np.load(f'samples/flow/{tid}_WT.npy')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    
    mut_dict = {}
    for mut in target.mutations:
        p = f'samples/flow/{tid}_{mut.id}.npy'
        if os.path.exists(p):
            mut_dict[mut.id] = np.load(p)
    
    fig = plot_mutation_comparison(
        wt, mut_dict, ref, target_name=target.name,
        save_path=f'samples/eval/{tid}_mutations.png'
    )
    fig.show()

In [ ]:
# 핵심 플롯 2: 응집 속도 상관관계
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

agg_rates = [r['agg_rate'] for r in mutation_results]
delta_rgs = [r['delta_rg_mean'] for r in mutation_results]
js_divs = [r['rg_js_wt_vs_mut'] for r in mutation_results]
labels = [f"{r['target']}\n{r['mutation']}" for r in mutation_results]

ax = axes[0]
ax.scatter(agg_rates, delta_rgs, s=80, c='steelblue', edgecolors='navy')
for i, lbl in enumerate(labels):
    ax.annotate(lbl, (agg_rates[i], delta_rgs[i]), fontsize=7,
                textcoords='offset points', xytext=(5, 5))
ax.set_xlabel('Relative Aggregation Rate')
ax.set_ylabel('\u0394Rg (mutant - WT)')
ax.set_title('Structural Change vs Aggregation Propensity')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
rho, pval = spearmanr(agg_rates, delta_rgs)
ax.text(0.05, 0.95, f'Spearman \u03c1={rho:.3f}\np={pval:.3f}',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax = axes[1]
ax.scatter(agg_rates, js_divs, s=80, c='coral', edgecolors='darkred')
for i, lbl in enumerate(labels):
    ax.annotate(lbl, (agg_rates[i], js_divs[i]), fontsize=7,
                textcoords='offset points', xytext=(5, 5))
ax.set_xlabel('Relative Aggregation Rate')
ax.set_ylabel('JS Divergence (WT vs Mutant Rg)')
ax.set_title('Ensemble Divergence vs Aggregation Propensity')
rho2, pval2 = spearmanr(agg_rates, js_divs)
ax.text(0.05, 0.95, f'Spearman \u03c1={rho2:.3f}\np={pval2:.3f}',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.tight_layout()
fig.savefig('samples/eval/aggregation_correlation.png', dpi=150)
fig.show()

print(f'\n*** Key Result ***')
print(f'\u0394Rg vs Aggregation Rate: \u03c1={rho:.3f}, p={pval:.3f}')
print(f'JS vs Aggregation Rate:  \u03c1={rho2:.3f}, p={pval2:.3f}')

In [ ]:
# Contact map 비교
from brain_idp_flow.eval.plots import plot_contact_maps

for tid, target in targets.items():
    wt = np.load(f'samples/flow/{tid}_WT.npy')
    ref = load_ped_or_fallback(target.ped_id, target.length)
    
    ensembles = {'PED ref': ref, 'FM WT': wt}
    if target.mutations:
        top_mut = max(target.mutations, key=lambda m: m.agg_rate_relative)
        p = f'samples/flow/{tid}_{top_mut.id}.npy'
        if os.path.exists(p):
            ensembles[f'FM {top_mut.id}'] = np.load(p)
    
    fig = plot_contact_maps(
        ensembles, title=f'{target.name} — Contact Frequency',
        save_path=f'samples/eval/{tid}_contacts.png'
    )
    fig.show()

In [ ]:
# 결과 저장 + Drive 백업
import json

summary = {
    'mutation_effects': mutation_results,
    'correlation': {
        'delta_rg_vs_agg': {'spearman_rho': float(rho), 'p_value': float(pval)},
        'js_vs_agg': {'spearman_rho': float(rho2), 'p_value': float(pval2)},
    }
}

with open('samples/eval/full_results.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

# Drive에 결과 백업
import shutil
result_backup = '/content/drive/MyDrive/brain_idp_flow_results'
if os.path.exists('/content/drive/MyDrive'):
    shutil.copytree('samples/eval', result_backup, dirs_exist_ok=True)
    print(f'Results backed up to {result_backup}')

print('\n=== All Done ===')